# 47 — Exact recording-batch CatBoost

**Decision:** retain the complete-identity CatBoost. Exact recording date adds 0.021 points to the direct vote, but misses the promotion gate and does not improve the existing archive synthesis.

In [1]:
from pathlib import Path
import json
import pandas as pd
PROJECT_DIR = Path('../..').resolve()
OUTPUT_DIR = PROJECT_DIR / '.runtime' / 'recording-batch-catboost-screen'
summary = pd.read_csv(OUTPUT_DIR / 'candidate-summary.csv', index_col='candidate')
support = pd.read_csv(OUTPUT_DIR / 'fold-support-audit.csv')
result = json.loads((OUTPUT_DIR / 'result.json').read_text(encoding='utf-8'))
result

{'engineered_features': 36,
 'categorical_features': 22,
 'unique_development_dates': 351,
 'batch_vote_accuracy': 0.8176136363636364,
 'batch_bag_accuracy': 0.81746632996633,
 'best_change_vs_promoted_identity_vote': 0.0004629629629628873,
 'passes_gate': False,
 'local_test_opened': False,
 'competition_predictions_generated': False,
 'external_evidence': 'https://github.com/drivendataorg/pump-it-up/tree/master/benedekrozemberczki'}

## Course-aligned lifecycle

1. **Define the goal and scope** — test repeated recording dates as survey-batch identities.
2. **Gather the data** — reuse complete supplied dates and the frozen development partition.
3. **Explore the data** — audit outer-training support for validation dates.
4. **Clean and preprocess the data** — preserve the strict complete-date contract.
5. **Select and engineer features** — append one exact ISO date category to complete identities.
6. **Define the machine-learning task** — retain three-class accuracy.
7. **Partition the data** — reuse frozen folds and nested stopping.
8. **Select and train candidate methods** — fit one depth-8 CatBoost and fixed votes.
9. **Evaluate and interpret the results** — compare direct and archive-synthesis substitutions.
10. **Deploy and iterate** — reject promotion and stop calendar variants.

In [2]:
support.style.format({
    'validation_unseen_share': '{:.3%}',
    'validation_support_below_20_share': '{:.3%}',
})

,fold,training_dates,validation_dates,validation_unseen_share,validation_support_below_20_share
0,1,343,317,0.084%,1.862%
1,2,342,320,0.095%,1.894%
2,3,347,316,0.042%,1.694%
3,4,347,314,0.042%,2.083%
4,5,345,320,0.063%,1.599%


In [3]:
summary.style.format({
    'mean_accuracy': '{:.4%}',
    'leader_change': '{:+.4%}',
    'worst_fold_change': '{:+.4%}',
    'repair_recall_change': '{:+.4%}',
})

,mean_accuracy,leader_change,fold_wins_vs_leader,worst_fold_change,repair_recall_change,passes_gate
candidate,,,,,,
archive_identity_vote,81.7929%,+0.0526%,5,+0.0105%,-0.2026%,False
archive_recording_batch_vote,81.7866%,+0.0463%,3,-0.0316%,-0.1739%,False
archive_identity_recording_batch_bag,81.7761%,+0.0358%,3,-0.0210%,-0.2606%,False
recording_batch_vote,81.7614%,+0.0210%,3,-0.0737%,-0.1161%,False
identity_recording_batch_bag,81.7466%,+0.0063%,3,-0.0210%,-0.1159%,False
promoted_identity_vote,81.7403%,+0.0000%,0,+0.0000%,+0.0000%,False
complete_identity_catboost,80.6229%,-1.1174%,0,-1.5888%,-5.8489%,False
recording_batch_catboost,80.5787%,-1.1616%,0,-1.7992%,-6.4282%,False


## Interpretation

The date category has sufficient fold support and makes a few useful ensemble corrections, but the standalone CatBoost is weaker. The existing archive synthesis remains better, indicating that spatial and frequency representations already absorb most survey-batch structure. No local test is opened.